In [1]:
"""
gan=genarative adversarial network
fashion mnist veri seti ile moda ürünü tasarımı

ganlar iki adet fonksiyon içeriyor
1-genrator aldıgı bir gürültüyü veriye cevirir
2-discriminator generatorun ürettiği fake görüntüyü alır ve gercek resmi
ele alır fake mi gercek mi karar verir

fashion mnist 10 farklı sınıfa sahip thsitt,sneaker,coat,sandal tarzı


"""

'\ngan=genarative adversarial network\nfashion mnist veri seti ile moda ürünü tasarımı\n\nganlar iki adet fonksiyon içeriyor\n1-genrator aldıgı bir gürültüyü veriye cevirir\n2-discriminator generatorun ürettiği fake görüntüyü alır ve gercek resmi\nele alır fake mi gercek mi karar verir\n\n\n'

In [15]:
#import libraries
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np 
import matplotlib.pyplot as plt
import os
from tensorflow.keras.datasets import fashion_mnist
BUFFER_SIZE=60000
BATCH_SIZE=32
NOISE_DIM=100 #generatore verilecek gürültü vektörünün boyutu 

In [10]:
(X_train,y_train),(X_test,y_test)=fashion_mnist.load_data()
#burada hepsini aldık amncak sadece X_train kullanacagız

In [46]:
X_train = X_train.reshape(-1, 28, 28, 1).astype("float32")
X_train = (X_train - 127.5) / 127.5
#şekillendirip floata cevirdik
#sonda ki 1 tf'in istedigi
#-1 ise kaç resim olmadıgını bilmedigimiz için yazdık

In [47]:
#veriyi karıstırma
train_dataset=tf.data.Dataset.from_tensor_slices(X_train).shuffle(BUFFER_SIZE).batch(BATCH_SIZE)

In [48]:
#gan modelini tanımlama 2 model barındırıyor generator model tanımalayalım
def build_generator_model():
    model=tf.keras.Sequential([
    layers.Dense(7*7*256,use_bias=False,input_shape=(NOISE_DIM,)),#gürültüyü özelllik haritasına dönüştercel
    layers.BatchNormalization(), #egitimi stabilize eder
    layers.LeakyReLU(), #activasyon fonksiyonu bu negatif degerleride alır
    layers.Reshape((7,7,256)), #tek boyutlu vektoru 3D ye donustur
    layers.Conv2DTranspose(128,(5,5),strides=(1,1),padding="same",use_bias=False),
    layers.BatchNormalization(),
    layers.LeakyReLU(),
    layers.Conv2DTranspose(64,(5,5),strides=(2,2),padding="same",use_bias=False),
    layers.BatchNormalization(),
    layers.LeakyReLU(),    
    layers.Conv2DTranspose(1,(5,5),strides=(2,2),padding="same",use_bias=False,activation="tanh")

        
    ])
    return model
    

In [49]:
def discriminator_model():
    model=tf.keras.Sequential([
        layers.Conv2D(64,(5,5),strides=(2,2),padding="same",input_shape=(28,28,1)),
        layers.LeakyReLU(),
        layers.Dropout(0.3),
        layers.Conv2D(128,(5,5),strides=(2,2),padding="same"),
        layers.LeakyReLU(),
        layers.Dropout(0.3),
        layers.Flatten(),#3D düzleştirme
        layers.Dense(1)
    
        
    ])
    return model

In [50]:
cross_entropy=tf.keras.losses.BinaryCrossentropy()

def disc_loss(real_output,fake_output):
    real_loss=cross_entropy(tf.ones_like(real_output),real_output) #gercek görüntüler için 1
    fake_loss=cross_entropy(tf.zeros_like(fake_output),fake_output) #saheteler için 0
    return real_loss+fake_loss #toplam kayıp

def gen_loss(fake_output):
    return cross_entropy(tf.ones_like(fake_output),fake_output) #gen sahte göötünyü 1 gibi göstericek

generator=build_generator_model()
disciminator=discriminator_model()
    

C:\Users\Ömer faruk\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
C:\Users\Ömer faruk\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [51]:
generator_opt=tf.keras.optimizers.Adam(1e-4)
disc_opt=tf.keras.optimizers.Adam(1e-4)

In [52]:
seed=tf.random.normal([16,NOISE_DIM]) #sabit gürültü örneği

In [56]:
def generate_and_save_image(model,epoch,test_input):
    predict=model(test_input,training=False)
    fig=plt.figure(figsize=(4,4))

    for i in range(predict.shape[0]):
        plt.subplot(4,4,i+1)
        plt.imshow((predict[i,:,:,0]+1)/2,cmap="gray")
        plt.axis("off")

    if not os.path.exists("generated_images"):
        os.makedirs("generated_images")

    plt.savefig(f"generated_images/image_At_epoch{epoch:03d}.png")
    plt.close()

In [57]:
def train(dataset, epochs):
    for epoch in range(1, epochs + 1):
        gen_loss_total = 0
        disc_loss_total = 0
        batch_count = 0

        for image_batch in dataset:
            noise = tf.random.normal([BATCH_SIZE, NOISE_DIM])

            with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
                # Yazım hatası düzeltildi: generated_images
                generated_images = generator(noise, training=True)
                
                real_output = disciminator(image_batch, training=True)
                # Yazım hatası düzeltildi: fake_output
                fake_output = disciminator(generated_images, training=True)
                
                # Değişken isimleri değiştirildi (fonksiyonlarla çakışmaması için)
                g_loss = gen_loss(fake_output)
                d_loss = disc_loss(real_output, fake_output)

            gradients_gen = gen_tape.gradient(g_loss, generator.trainable_variables)
            gradients_disc = disc_tape.gradient(d_loss, disciminator.trainable_variables)

            generator_opt.apply_gradients(zip(gradients_gen, generator.trainable_variables))
            disc_opt.apply_gradients(zip(gradients_disc, disciminator.trainable_variables))

            gen_loss_total += g_loss
            disc_loss_total += d_loss
            batch_count += 1

        print(f"Epoch: {epoch}/{epochs} | Gen Loss: {gen_loss_total/batch_count:.3f} | Disc Loss: {disc_loss_total/batch_count:.3f}")
        generate_and_save_image(generator, epoch, seed)

# KRİTİK DÜZELTME: X_train değil, train_dataset gönderiyoruz
train(train_dataset, 2)

Epoch: 1/2 | Gen Loss: 0.696 | Disc Loss: 1.391
Epoch: 2/2 | Gen Loss: 0.696 | Disc Loss: 1.390
